In [ ]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 18.9 MB/s eta 0:00:00


In [1]:
# -*- coding: utf-8 -*-
"""
╔══════════════════════════════════════════════════════════════╗
║     ADVANCED SQL QUERY GENERATOR - IBM Granite 3.3 2B       ║
║     Natural Language → Production-Ready SQL                 ║
╚══════════════════════════════════════════════════════════════╝
"""

# ── Install dependencies ──────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["gradio>=4.0", "transformers", "torch", "reportlab", "requests"]:
    try:
        install(pkg)
    except:
        pass

# ── Imports ───────────────────────────────────────────────────
import gradio as gr
from datetime import datetime
from pathlib import Path
import json, sqlite3, re, os, urllib.parse, traceback

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors

# ── IBM Granite Model (lazy load) ─────────────────────────────
_pipe = None

def get_model():
    global _pipe
    if _pipe is None:
        from transformers import pipeline
        print("⏳ Loading IBM Granite 3.3-2B …")
        _pipe = pipeline(
            "text-generation",
            model="ibm-granite/granite-3.3-2b-instruct",
            max_new_tokens=256,
            do_sample=False,
        )
        print("✅ Model loaded!")
    return _pipe

# ══════════════════════════════════════════════════════════════
#  DATABASE SCHEMA  (extended)
# ══════════════════════════════════════════════════════════════
SCHEMA = {
    "users": {
        "id": "INTEGER PRIMARY KEY",
        "name": "TEXT",
        "email": "TEXT",
        "signup_date": "DATE",
        "age": "INTEGER",
        "country": "TEXT",
        "status": "TEXT",
        "role": "TEXT",
    },
    "orders": {
        "id": "INTEGER PRIMARY KEY",
        "user_id": "INTEGER",
        "product_name": "TEXT",
        "amount": "REAL",
        "order_date": "DATE",
        "status": "TEXT",
        "quantity": "INTEGER",
    },
    "products": {
        "id": "INTEGER PRIMARY KEY",
        "name": "TEXT",
        "price": "REAL",
        "category": "TEXT",
        "stock": "INTEGER",
        "rating": "REAL",
        "created_at": "DATE",
    },
    "transactions": {
        "id": "INTEGER PRIMARY KEY",
        "user_id": "INTEGER",
        "amount": "REAL",
        "date": "DATE",
        "type": "TEXT",
        "status": "TEXT",
    },
    "employees": {
        "id": "INTEGER PRIMARY KEY",
        "name": "TEXT",
        "department": "TEXT",
        "salary": "REAL",
        "hire_date": "DATE",
        "manager_id": "INTEGER",
    },
    "inventory": {
        "id": "INTEGER PRIMARY KEY",
        "product_id": "INTEGER",
        "warehouse": "TEXT",
        "quantity": "INTEGER",
        "last_updated": "DATE",
    },
}

SCHEMA_DDL = "\n".join(
    f"CREATE TABLE {tbl} ({', '.join(f'{c} {t}' for c, t in cols.items())});"
    for tbl, cols in SCHEMA.items()
)

# ══════════════════════════════════════════════════════════════
#  SECURITY
# ══════════════════════════════════════════════════════════════
DANGEROUS = ["DROP", "DELETE", "ALTER", "TRUNCATE", "INSERT", "UPDATE",
             "EXEC", "EXECUTE", "GRANT", "REVOKE", "CREATE"]
DANGEROUS_CHARS = [";", "--", "/*", "*/", "xp_", "sp_"]

def is_safe(text: str) -> tuple[bool, str]:
    u = text.upper()
    for kw in DANGEROUS:
        if re.search(rf"\b{kw}\b", u):
            return False, f"Dangerous keyword detected: **{kw}**"
    for ch in DANGEROUS_CHARS:
        if ch in text:
            return False, f"Dangerous character sequence detected: `{ch}`"
    return True, ""

# ══════════════════════════════════════════════════════════════
#  RULE-BASED SQL ENGINE  (fast, offline)
# ══════════════════════════════════════════════════════════════
TABLE_KEYWORDS = {
    "users":        ["user", "users", "customer", "customers", "account", "member", "person"],
    "orders":       ["order", "orders", "purchase", "purchases", "buy", "bought", "sale"],
    "products":     ["product", "products", "item", "items", "goods", "catalog"],
    "transactions": ["transaction", "transactions", "payment", "payments", "transfer"],
    "employees":    ["employee", "employees", "staff", "worker", "hire", "department", "salary"],
    "inventory":    ["inventory", "stock", "warehouse", "supply"],
}

COLUMN_KEYWORDS = {
    "name":         ["name", "username", "full name"],
    "email":        ["email", "mail"],
    "signup_date":  ["signup", "registered", "joined"],
    "age":          ["age", "years old"],
    "amount":       ["amount", "total", "cost", "price", "revenue"],
    "order_date":   ["order date", "ordered"],
    "status":       ["status", "state"],
    "country":      ["country", "location", "region"],
    "department":   ["department", "dept", "team"],
    "salary":       ["salary", "pay", "wage"],
    "category":     ["category", "type", "kind"],
    "stock":        ["stock", "inventory", "quantity"],
    "rating":       ["rating", "score", "stars"],
}

AGG_KEYWORDS = {
    "COUNT":   ["count", "how many", "number of", "total count"],
    "SUM":     ["sum", "total", "revenue", "earnings"],
    "AVG":     ["average", "avg", "mean"],
    "MAX":     ["max", "maximum", "highest", "most", "top"],
    "MIN":     ["min", "minimum", "lowest", "least"],
}

def detect_table(q: str) -> str:
    ql = q.lower()
    for tbl, kws in TABLE_KEYWORDS.items():
        for kw in kws:
            if kw in ql:
                return tbl
    return "users"

def detect_aggregation(q: str) -> tuple[str, str] | None:
    ql = q.lower()
    for fn, kws in AGG_KEYWORDS.items():
        for kw in kws:
            if kw in ql:
                # Pick the most relevant column
                for col, ckws in COLUMN_KEYWORDS.items():
                    for ckw in ckws:
                        if ckw in ql:
                            return fn, col
                return fn, "*"
    return None

def detect_columns(table: str, q: str) -> list[str]:
    ql = q.lower()
    cols = list(SCHEMA[table].keys())
    selected = []
    for col, kws in COLUMN_KEYWORDS.items():
        if col in cols:
            for kw in kws:
                if kw in ql:
                    selected.append(col)
                    break
    return list(dict.fromkeys(selected)) or ["*"]

def detect_conditions(table: str, q: str) -> list[str]:
    ql = q.lower()
    conds = []
    cols = SCHEMA[table]

    # ── date shortcuts ──
    date_col = "signup_date" if "signup_date" in cols else ("order_date" if "order_date" in cols else "date" if "date" in cols else None)
    if date_col:
        if "today" in ql:
            conds.append(f"DATE({date_col}) = DATE('now')")
        elif "last week" in ql:
            conds.append(f"DATE({date_col}) >= DATE('now','-7 days')")
        elif "last month" in ql:
            conds.append(f"DATE({date_col}) >= DATE('now','-1 month')")
        elif "last year" in ql or "this year" in ql:
            conds.append(f"strftime('%Y',{date_col}) = strftime('%Y','now')")

    # ── status ──
    if "status" in cols:
        if "active" in ql and "inactive" not in ql:
            conds.append("status = 'active'")
        elif "inactive" in ql or "pending" in ql:
            conds.append(f"status = '{'inactive' if 'inactive' in ql else 'pending'}'")

    # ── numeric filters ──
    num_col = "amount" if "amount" in cols else ("salary" if "salary" in cols else ("price" if "price" in cols else None))
    if num_col:
        m = re.search(r"(?:more than|greater than|above|over)\s*\$?([0-9]+(?:\.[0-9]+)?)", ql)
        if m:
            conds.append(f"{num_col} > {m.group(1)}")
        m = re.search(r"(?:less than|below|under)\s*\$?([0-9]+(?:\.[0-9]+)?)", ql)
        if m:
            conds.append(f"{num_col} < {m.group(1)}")

    # ── country / department ──
    for col in ["country", "department", "category", "warehouse"]:
        if col in cols:
            m = re.search(rf"{col}\s+(?:is\s+)?['\"]?([a-zA-Z\s]+)['\"]?", ql)
            if m:
                conds.append(f"{col} = '{m.group(1).strip().title()}'")

    # ── age ──
    if "age" in cols:
        m = re.search(r"age\s*(?:>|>=|<|<=|=)\s*([0-9]+)", ql)
        if m:
            op = re.search(r"(>|>=|<|<=|=)", ql[ql.find("age"):ql.find("age")+10])
            if op:
                conds.append(f"age {op.group()} {m.group(1)}")

    return conds

def detect_order(table: str, q: str) -> str | None:
    ql = q.lower()
    cols = SCHEMA[table]
    direction = "DESC" if any(w in ql for w in ["highest", "most", "top", "latest", "newest", "recent"]) else "ASC"
    if any(w in ql for w in ["latest", "newest", "recent"]):
        for dc in ["signup_date", "order_date", "date", "created_at", "hire_date"]:
            if dc in cols:
                return f"ORDER BY {dc} {direction}"
    if any(w in ql for w in ["highest", "most expensive", "top earning"]):
        for nc in ["amount", "salary", "price", "rating"]:
            if nc in cols:
                return f"ORDER BY {nc} {direction}"
    if "order by" in ql:
        for col in cols:
            if col in ql:
                return f"ORDER BY {col}"
    return None

def detect_limit(q: str) -> str:
    ql = q.lower()
    m = re.search(r"\b(?:top|first|limit)\s+([0-9]+)\b", ql)
    if m:
        return f"LIMIT {m.group(1)}"
    if "all" in ql:
        return ""
    return "LIMIT 10"

def detect_group_by(table: str, q: str) -> str | None:
    ql = q.lower()
    cols = SCHEMA[table]
    for kw in ["group by", "per ", "each ", "by country", "by department", "by category", "by status"]:
        if kw in ql:
            for col in ["country", "department", "category", "status", "type"]:
                if col in cols:
                    return f"GROUP BY {col}"
    return None

def rule_based_sql(q: str) -> tuple[str, str]:
    safe, reason = is_safe(q)
    if not safe:
        return "", f"🚫 {reason}"

    table = detect_table(q)
    agg = detect_aggregation(q)

    if agg:
        fn, col = agg
        select = f"{fn}({col}) AS result"
        group = detect_group_by(table, q)
        if group:
            grp_col = group.split()[-1]
            if grp_col in SCHEMA[table]:
                select = f"{grp_col}, {select}"
    else:
        cols = detect_columns(table, q)
        select = ", ".join(cols)
        group = None

    sql = f"SELECT {select}\nFROM {table}"

    conds = detect_conditions(table, q)
    if conds:
        sql += "\nWHERE " + "\n  AND ".join(conds)

    if group:
        sql += f"\n{group}"

    order = detect_order(table, q)
    if order:
        sql += f"\n{order}"

    lim = detect_limit(q)
    if lim:
        sql += f"\n{lim}"

    sql += ";"
    note = f"Table: **{table}** | Columns: `{select}`"
    return sql, note

# ══════════════════════════════════════════════════════════════
#  GRANITE AI SQL ENGINE
# ══════════════════════════════════════════════════════════════
SYSTEM_PROMPT = f"""You are an expert SQL assistant. Convert natural language to SQLite SQL.

Database schema:
{SCHEMA_DDL}

Rules:
- Only generate SELECT queries
- Use exact table/column names from schema
- Output ONLY the SQL query, no explanation
- End with semicolon
- Use SQLite syntax (DATE('now'), strftime, etc.)
"""

def granite_sql(q: str) -> tuple[str, str]:
    try:
        pipe = get_model()
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Convert to SQL: {q}"},
        ]
        result = pipe(messages)[0]["generated_text"]
        # Extract last assistant message
        if isinstance(result, list):
            sql = result[-1].get("content", "")
        else:
            sql = str(result)

        # Clean up
        sql = re.sub(r"```sql|```", "", sql).strip()
        if not sql.endswith(";"):
            sql += ";"

        safe, reason = is_safe(sql)
        if not safe:
            return "", f"🚫 AI generated unsafe SQL: {reason}"

        return sql, "🤖 Generated by **IBM Granite 3.3-2B**"
    except Exception as e:
        return "", f"⚠️ Granite error: {str(e)[:120]}"

# ══════════════════════════════════════════════════════════════
#  IN-MEMORY DEMO DATABASE
# ══════════════════════════════════════════════════════════════
def get_demo_db():
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.executescript("""
        CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, email TEXT,
            signup_date DATE, age INTEGER, country TEXT, status TEXT, role TEXT);
        INSERT INTO users VALUES
            (1,'Alice','alice@ex.com','2024-01-15',28,'India','active','admin'),
            (2,'Bob','bob@ex.com','2024-03-22',35,'USA','inactive','user'),
            (3,'Carol','carol@ex.com','2025-11-05',24,'UK','active','user'),
            (4,'David','david@ex.com','2025-12-18',42,'India','active','manager'),
            (5,'Eva','eva@ex.com','2026-01-10',31,'Germany','active','user');

        CREATE TABLE orders (id INTEGER PRIMARY KEY, user_id INTEGER,
            product_name TEXT, amount REAL, order_date DATE, status TEXT, quantity INTEGER);
        INSERT INTO orders VALUES
            (1,1,'Laptop',1200.00,'2025-12-01','completed',1),
            (2,2,'Phone',800.00,'2025-12-15','pending',2),
            (3,3,'Tablet',450.00,'2026-01-05','completed',1),
            (4,1,'Monitor',350.00,'2026-02-10','shipped',2),
            (5,4,'Keyboard',75.00,'2026-02-20','completed',3);

        CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL,
            category TEXT, stock INTEGER, rating REAL, created_at DATE);
        INSERT INTO products VALUES
            (1,'Laptop',1200.00,'Electronics',50,4.5,'2024-06-01'),
            (2,'Phone',800.00,'Electronics',120,4.3,'2024-07-15'),
            (3,'Tablet',450.00,'Electronics',80,4.1,'2024-08-20'),
            (4,'Desk Chair',250.00,'Furniture',30,4.7,'2024-09-10'),
            (5,'Notebook',15.00,'Stationery',500,4.0,'2024-10-05');

        CREATE TABLE transactions (id INTEGER PRIMARY KEY, user_id INTEGER,
            amount REAL, date DATE, type TEXT, status TEXT);
        INSERT INTO transactions VALUES
            (1,1,1200.00,'2025-12-01','payment','completed'),
            (2,2,800.00,'2025-12-15','payment','pending'),
            (3,3,450.00,'2026-01-05','refund','completed'),
            (4,1,350.00,'2026-02-10','payment','completed'),
            (5,5,200.00,'2026-02-25','transfer','completed');

        CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, department TEXT,
            salary REAL, hire_date DATE, manager_id INTEGER);
        INSERT INTO employees VALUES
            (1,'John','Engineering',95000,'2020-03-15',NULL),
            (2,'Sara','Marketing',75000,'2021-07-01',1),
            (3,'Mike','Engineering',88000,'2022-01-10',1),
            (4,'Lisa','HR',65000,'2019-11-20',NULL),
            (5,'Tom','Sales',70000,'2023-05-05',2);

        CREATE TABLE inventory (id INTEGER PRIMARY KEY, product_id INTEGER,
            warehouse TEXT, quantity INTEGER, last_updated DATE);
        INSERT INTO inventory VALUES
            (1,1,'Chennai',20,'2026-02-01'),
            (2,2,'Mumbai',50,'2026-02-05'),
            (3,3,'Delhi',30,'2026-02-10'),
            (4,4,'Bangalore',15,'2026-02-12'),
            (5,5,'Hyderabad',200,'2026-02-15');
    """)
    conn.commit()
    return conn

def execute_sql(sql: str) -> tuple[list, list] | tuple[None, str]:
    """Run SQL on demo DB, return (rows, headers) or (None, error)"""
    try:
        conn = get_demo_db()
        cur = conn.cursor()
        cur.execute(sql.rstrip(";"))
        rows = cur.fetchall()
        headers = [d[0] for d in cur.description] if cur.description else []
        conn.close()
        return rows, headers
    except Exception as e:
        return None, str(e)

# ══════════════════════════════════════════════════════════════
#  STATE
# ══════════════════════════════════════════════════════════════
query_history: list[dict] = []

# ══════════════════════════════════════════════════════════════
#  CORE HANDLER
# ══════════════════════════════════════════════════════════════
def process_query(user_query: str, use_granite: bool, chat_history: list):
    if not user_query.strip():
        return chat_history, "", ""

    # Generate SQL
    if use_granite:
        sql, note = granite_sql(user_query)
        engine = "IBM Granite AI"
    else:
        sql, note = rule_based_sql(user_query)
        engine = "Rule Engine"

    if not sql:
        msg = f"### ❌ Error\n{note}"
        chat_history.append({"role": "user", "content": user_query})
        chat_history.append({"role": "assistant", "content": msg})
        return chat_history, "", ""

    # Execute on demo DB
    rows, headers = execute_sql(sql)

    # Build markdown table
    result_md = ""
    if rows is None:
        result_md = f"\n\n⚠️ **Execution note:** `{headers}`"
    elif rows:
        header_row = " | ".join(str(h) for h in headers)
        sep_row = " | ".join("---" for _ in headers)
        data_rows = "\n".join(" | ".join(str(c) for c in row) for row in rows[:10])
        result_md = f"\n\n**Results (demo data):**\n\n| {header_row} |\n| {sep_row} |\n" + \
                    "\n".join(f"| {' | '.join(str(c) for c in row)} |" for row in rows[:10])
    else:
        result_md = "\n\n*No rows matched in demo data.*"

    response = f"""### ✅ SQL Generated — {engine}

```sql
{sql}
```

📌 {note}{result_md}"""

    query_history.append({
        "user": user_query,
        "sql": sql,
        "engine": engine,
        "note": note,
        "timestamp": datetime.now().isoformat(),
    })

    chat_history.append({"role": "user", "content": user_query})
    chat_history.append({"role": "assistant", "content": response})
    return chat_history, "", sql

def clear_all():
    global query_history
    query_history = []
    return [], "", ""

def get_stats():
    total = len(query_history)
    granite = sum(1 for q in query_history if q["engine"] == "IBM Granite AI")
    rule = total - granite
    tables = {}
    for q in query_history:
        tbl = detect_table(q["user"])
        tables[tbl] = tables.get(tbl, 0) + 1
    top_table = max(tables, key=tables.get) if tables else "—"
    return f"""### 📊 Session Statistics

| Metric | Value |
|---|---|
| Total Queries | **{total}** |
| Granite AI Queries | **{granite}** |
| Rule Engine Queries | **{rule}** |
| Most Queried Table | **{top_table}** |
| Session Start | {query_history[0]['timestamp'][:19] if query_history else '—'} |
"""

def validate_sql_only(sql_text: str) -> str:
    if not sql_text.strip():
        return "⬆️ Enter SQL above to validate."
    safe, reason = is_safe(sql_text)
    if not safe:
        return f"🚫 **UNSAFE** — {reason}"
    try:
        conn = get_demo_db()
        conn.execute(f"EXPLAIN QUERY PLAN {sql_text.rstrip(';')}")
        conn.close()
        return "✅ **Valid SQL** — Syntax OK, no dangerous operations detected."
    except Exception as e:
        return f"⚠️ **Syntax Error** — {str(e)}"

# ══════════════════════════════════════════════════════════════
#  EXPORT
# ══════════════════════════════════════════════════════════════
def export_pdf():
    if not query_history:
        return "❌ No queries to export.", None
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"/tmp/sql_report_{ts}.pdf"
    doc = SimpleDocTemplate(path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    title_style = ParagraphStyle("T", parent=styles["Heading1"],
                                  fontSize=22, textColor=colors.HexColor("#00D4AA"),
                                  spaceAfter=20, alignment=1)
    story.append(Paragraph("SQL Query Generator Report", title_style))
    story.append(Paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles["Normal"]))
    story.append(Spacer(1, 0.2*inch))

    for i, q in enumerate(query_history, 1):
        story.append(Paragraph(f"<b>Query #{i} — {q['engine']}</b>", styles["Heading2"]))
        story.append(Paragraph(f"<b>Request:</b> {q['user']}", styles["Normal"]))
        story.append(Paragraph(f"<b>SQL:</b> <font color='#0066CC'>{q['sql']}</font>", styles["Normal"]))
        story.append(Paragraph(f"<b>Time:</b> {q['timestamp'][:19]}", styles["Normal"]))
        story.append(Spacer(1, 0.15*inch))

    doc.build(story)
    return f"✅ PDF ready: `sql_report_{ts}.pdf`", path

def export_json():
    if not query_history:
        return "❌ No queries.", None
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"/tmp/sql_queries_{ts}.json"
    Path(path).write_text(json.dumps(query_history, indent=2))
    return f"✅ JSON ready: `sql_queries_{ts}.json`", path

def export_txt():
    if not query_history:
        return "❌ No queries.", None
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"/tmp/sql_queries_{ts}.txt"
    lines = [f"SQL Query Report | {datetime.now()}\n{'='*60}\n"]
    for i, q in enumerate(query_history, 1):
        lines += [f"Query #{i} [{q['engine']}]",
                  f"Request : {q['user']}",
                  f"SQL     : {q['sql']}",
                  f"Time    : {q['timestamp'][:19]}",
                  "-"*60, ""]
    Path(path).write_text("\n".join(lines))
    return f"✅ TXT ready: `sql_queries_{ts}.txt`", path

def whatsapp_share():
    if not query_history:
        return "❌ No queries to share!"
    msg = "🔍 *SQL Queries*\n\n"
    for i, q in enumerate(query_history[-5:], 1):
        msg += f"*{i}.* {q['user']}\n`{q['sql']}`\n\n"
    link = f"https://wa.me/?text={urllib.parse.quote(msg)}"
    return f"**WhatsApp Link:**\n{link}"

# ══════════════════════════════════════════════════════════════
#  SAMPLE QUERIES
# ══════════════════════════════════════════════════════════════
SAMPLES = [
    "Show me all active users from India",
    "Get the top 5 orders with highest amount",
    "Count users by country",
    "Show products with rating above 4.3",
    "List employees in Engineering department",
    "Find all transactions completed this year",
    "Show inventory in Chennai warehouse",
    "Get average salary per department",
    "Show users who signed up last month",
    "Find all pending orders",
]

# ══════════════════════════════════════════════════════════════
#  GRADIO UI
# ══════════════════════════════════════════════════════════════
CSS = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Syne:wght@400;600;800&display=swap');

:root {
    --bg:        #0a0e1a;
    --surface:   #111827;
    --surface2:  #1a2235;
    --accent:    #00D4AA;
    --accent2:   #FF6B6B;
    --accent3:   #7C6FED;
    --text:      #E2E8F0;
    --muted:     #64748B;
    --border:    #1e2d45;
    --glow:      0 0 20px rgba(0,212,170,0.15);
}

* { box-sizing: border-box; }
body, .gradio-container { background: var(--bg) !important; font-family: 'Syne', sans-serif !important; color: var(--text) !important; }

.gr-panel, .gr-box, .panel { background: var(--surface) !important; border: 1px solid var(--border) !important; border-radius: 12px !important; }
.gr-input, .gr-textarea, textarea, input { background: var(--surface2) !important; border: 1px solid var(--border) !important; color: var(--text) !important; font-family: 'Syne', sans-serif !important; border-radius: 8px !important; }
.gr-input:focus, textarea:focus { border-color: var(--accent) !important; box-shadow: var(--glow) !important; }

.gr-button { font-family: 'Syne', sans-serif !important; font-weight: 600 !important; border-radius: 8px !important; transition: all 0.2s !important; cursor: pointer !important; }
.gr-button-primary { background: linear-gradient(135deg, var(--accent), #00a882) !important; border: none !important; color: #0a0e1a !important; }
.gr-button-primary:hover { transform: translateY(-1px) !important; box-shadow: var(--glow) !important; }
.gr-button-secondary { background: var(--surface2) !important; border: 1px solid var(--border) !important; color: var(--text) !important; }
.gr-button-stop { background: linear-gradient(135deg, var(--accent2), #d94f4f) !important; border: none !important; color: white !important; }

.chatbot { background: var(--surface) !important; border-radius: 12px !important; }
.message.user { background: var(--surface2) !important; border-left: 3px solid var(--accent3) !important; }
.message.bot { background: var(--surface2) !important; border-left: 3px solid var(--accent) !important; }

label { color: var(--accent) !important; font-weight: 600 !important; font-size: 0.82rem !important; letter-spacing: 0.06em !important; text-transform: uppercase !important; }

.schema-pill { display:inline-block; background:var(--surface2); border:1px solid var(--border); border-radius:20px; padding:3px 10px; font-family:'JetBrains Mono',monospace; font-size:0.75rem; color:var(--accent); margin:2px; }

#title-block { text-align:center; padding:24px 0 8px; }
#title-block h1 { font-family:'Syne',sans-serif; font-weight:800; font-size:2.4rem; background: linear-gradient(135deg, var(--accent), var(--accent3)); -webkit-background-clip:text; -webkit-text-fill-color:transparent; margin:0; }
#title-block p { color:var(--muted); font-size:0.95rem; margin-top:6px; }

code, pre { font-family:'JetBrains Mono',monospace !important; background:var(--bg) !important; color:var(--accent) !important; border-radius:6px !important; padding:2px 6px !important; font-size:0.85rem !important; }

.toggle-label { color: var(--accent3) !important; }
"""

with gr.Blocks(title="SQL Query Generator — IBM Granite", css=CSS, theme=gr.themes.Base()) as demo:

    # ── Header ──────────────────────────────────────────────
    gr.HTML("""
    <div id="title-block">
        <h1>⚡ SQL Query Generator</h1>
        <p>Natural Language → Production-Ready SQLite SQL &nbsp;|&nbsp; Powered by IBM Granite 3.3-2B</p>
    </div>
    """)

    with gr.Tabs():

        # ── Tab 1 : Query ──────────────────────────────────
        with gr.Tab("🔍 Query Generator"):
            with gr.Row():

                # Left: chat
                with gr.Column(scale=3):
                    chatbot = gr.Chatbot(
                        label="Conversation",
                        height=460,
                        show_copy_button=True,
                        type="messages",
                        avatar_images=(None, "https://huggingface.co/ibm-granite/granite-3.3-2b-instruct/resolve/main/granite.png"),
                    )
                    with gr.Row():
                        query_input = gr.Textbox(
                            placeholder="e.g. Show top 5 active users from India ordered by signup date...",
                            label="Natural Language Request",
                            lines=2, scale=5,
                        )
                        send_btn = gr.Button("Generate →", variant="primary", scale=1, min_width=120)

                    with gr.Row():
                        use_granite = gr.Checkbox(
                            label="🤖 Use IBM Granite AI  (uncheck = fast rule engine)",
                            value=False, elem_classes=["toggle-label"]
                        )
                        clear_btn = gr.Button("🗑 Clear", variant="stop", scale=1, min_width=80)

                    gr.Markdown("**💡 Sample Queries:**")
                    with gr.Row():
                        for s in SAMPLES[:5]:
                            gr.Button(s, size="sm").click(
                                lambda x=s: x, outputs=query_input
                            )
                    with gr.Row():
                        for s in SAMPLES[5:]:
                            gr.Button(s, size="sm").click(
                                lambda x=s: x, outputs=query_input
                            )

                # Right: info
                with gr.Column(scale=1, min_width=220):
                    gr.Markdown("### 📋 Schema")
                    gr.HTML("""
                    <div style="line-height:2">
                    """ + "".join(f'<span class="schema-pill">{t}</span>' for t in SCHEMA) + """
                    </div>
                    <hr style="border-color:#1e2d45; margin:12px 0">
                    <div style="font-size:0.8rem; color:#64748B">
                    <b style="color:#00D4AA">6 Tables</b><br>
                    users · orders · products<br>
                    transactions · employees · inventory
                    </div>
                    """)

                    gr.Markdown("### 🔒 Security")
                    gr.HTML("""
                    <div style="font-size:0.8rem; color:#64748B; line-height:1.8">
                    ✅ SELECT only<br>
                    🚫 No DROP/DELETE/ALTER<br>
                    🔍 SQL injection blocked<br>
                    ⚡ Live demo execution
                    </div>
                    """)

                    stats_btn = gr.Button("📊 Show Stats", size="sm", variant="secondary")
                    stats_out = gr.Markdown("")
                    stats_btn.click(get_stats, outputs=stats_out)

            sql_output = gr.Textbox(label="Last Generated SQL", lines=3, interactive=False)

            # Events
            sql_state = gr.State("")
            send_btn.click(process_query, [query_input, use_granite, chatbot], [chatbot, query_input, sql_output])
            query_input.submit(process_query, [query_input, use_granite, chatbot], [chatbot, query_input, sql_output])
            clear_btn.click(clear_all, outputs=[chatbot, query_input, sql_output])

        # ── Tab 2 : SQL Validator ──────────────────────────
        with gr.Tab("🛡 SQL Validator"):
            gr.Markdown("### Paste any SQL to validate safety & syntax against demo schema")
            val_input = gr.Textbox(label="SQL to Validate", lines=6, placeholder="SELECT * FROM users WHERE status='active';")
            val_btn = gr.Button("Validate", variant="primary")
            val_result = gr.Markdown("")
            val_btn.click(validate_sql_only, val_input, val_result)
            val_input.change(validate_sql_only, val_input, val_result)

        # ── Tab 3 : Export ────────────────────────────────
        with gr.Tab("📥 Export & Share"):
            gr.Markdown("### Export your query history")
            with gr.Row():
                with gr.Column():
                    pdf_btn = gr.Button("📄 Export PDF", variant="primary")
                    pdf_status = gr.Markdown("")
                    pdf_file = gr.File(label="Download PDF")
                    pdf_btn.click(export_pdf, outputs=[pdf_status, pdf_file])

                with gr.Column():
                    json_btn = gr.Button("🗂 Export JSON", variant="secondary")
                    json_status = gr.Markdown("")
                    json_file = gr.File(label="Download JSON")
                    json_btn.click(export_json, outputs=[json_status, json_file])

                with gr.Column():
                    txt_btn = gr.Button("📋 Export TXT", variant="secondary")
                    txt_status = gr.Markdown("")
                    txt_file = gr.File(label="Download TXT")
                    txt_btn.click(export_txt, outputs=[txt_status, txt_file])

            gr.Markdown("---")
            gr.Markdown("### 📱 Share")
            wa_btn = gr.Button("💬 Generate WhatsApp Link", variant="secondary")
            share_out = gr.Markdown("")
            wa_btn.click(whatsapp_share, outputs=share_out)

        # ── Tab 4 : Help ──────────────────────────────────
        with gr.Tab("📖 Help & Docs"):
            gr.Markdown(f"""
## How to Use

### Two SQL Engines
| Engine | Speed | Accuracy | Requires GPU |
|--------|-------|----------|-------------|
| **Rule Engine** (default) | ⚡ Instant | Good for standard queries | No |
| **IBM Granite 3.3-2B** | 🐢 ~10s | Best for complex queries | Recommended |

### Supported Query Types
- **Filters:** `active users`, `orders above $500`, `last month signups`
- **Aggregations:** `count users by country`, `average salary per department`
- **Sorting:** `top 5 products by rating`, `latest transactions`
- **Multi-condition:** `active users from India who signed up this year`

### Database Schema
```
{SCHEMA_DDL}
```

### Security
All queries are validated:
- Only SELECT statements allowed
- No DROP, DELETE, ALTER, TRUNCATE, INSERT, UPDATE
- No semicolon injection, comment injection
- Executed on isolated in-memory SQLite (no real data affected)
""")

print("\n✅ Advanced SQL Query Generator ready!\n")
demo.launch(share=True, show_error=True, show_api=False)

/tmp/ipykernel_190/1803784770.py:675: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="SQL Query Generator — IBM Granite", css=CSS, theme=gr.themes.Base()) as demo:
/tmp/ipykernel_190/1803784770.py:675: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="SQL Query Generator — IBM Granite", css=CSS, theme=gr.themes.Base()) as demo:
/tmp/ipykernel_190/1803784770.py:693: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_190/1803784770.py:693: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you wan


✅ Advanced SQL Query Generator ready!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://096524eb9cdc4ef335.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
